In [ ]:
# Required Libraries
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, roc_curve, auc,classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from collections import Counter
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords  # Import stopwords
from nltk.stem import PorterStemmer
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
import json

In [ ]:
# Download stopwords if not already done
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Load dataset
df = pd.read_csv(r'reviews.csv')

In [ ]:
# Top 5 Records in dataset
df.head(5)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [ ]:
# Shape of the dataset
df.shape

(100000, 9)

In [ ]:
# Preprocess the text data by (Tokenization, Stop words, Stemming)
# Clean the text, convert to lowercase, remove non-alphabet characters, tokenize, apply stemming, and remove stopwords
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Remove non-alphabet characters
    tokens = word_tokenize(text)  # Tokenize text into words
    stop_words = set(stopwords.words('english'))  # Set of English stopwords
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    stemmer = PorterStemmer()  # Initialize the Porter Stemmer
    tokens = [stemmer.stem(word) for word in tokens]  # Apply stemming
    return ' '.join(tokens)  # Join tokens back into a string

In [ ]:
# New Coloumn in dataset that store processed dataset
df['processed_text'] = df['text'].apply(preprocess_text)  # Applying preprocessing to the reviews

In [ ]:
# Records in dataset
df.head(5)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,processed_text
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,decid eat awar go take hour begin end tri mult...
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18,ive taken lot spin class year noth compar clas...
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,famili diner buffet eclect assort larg chicken...
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,wow yummi differ delici favorit lamb curri kor...
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,cute interior owner gave us tour upcom patioro...


In [ ]:
# Showing original text, processed text
df[['text', 'processed_text']]

,text,processed_text
0,"If you decide to eat here, just be aware it is...",decid eat awar go take hour begin end tri mult...
1,I've taken a lot of spin classes over the year...,ive taken lot spin class year noth compar clas...
2,Family diner. Had the buffet. Eclectic assortm...,famili diner buffet eclect assort larg chicken...
3,"Wow! Yummy, different, delicious. Our favo...",wow yummi differ delici favorit lamb curri kor...
4,Cute interior and owner (?) gave us tour of up...,cute interior owner gave us tour upcom patioro...
...,...,...
99995,Came here for lunch with a group. They were bu...,came lunch group busi still room us servic goo...
99996,The equipment is so old and so felty! I just u...,equip old felti upgrad multi club membership c...
99997,This is one of my favorite Mexican restaurants...,one favorit mexican restaur authent menu typic...
99998,Came here for brunch - had an omlette ($19 + t...,came brunch omlett tax tip food wayyyyyyy over...


In [ ]:
# Label encoding (Positive, Negative)
# We label 'Negative' for <= 2 stars, 'Positive' for 4 or 5 stars, 'Neutral' for 3 stars
def encode_labels(stars):
    if stars <= 2:
        return 'Negative'
    elif stars == 3:
        return 'Neutral'
    else:
        return 'Positive'

In [ ]:
# New Column in dataset to Store lablel's
df['label'] = df['stars'].apply(encode_labels)  # Apply the label encoding function

In [ ]:
# show dataset
df[['stars', 'label', 'text', 'processed_text']]

,stars,label,text,processed_text
0,3,Negative,"If you decide to eat here, just be aware it is...",decid eat awar go take hour begin end tri mult...
1,5,Positive,I've taken a lot of spin classes over the year...,ive taken lot spin class year noth compar clas...
2,3,Negative,Family diner. Had the buffet. Eclectic assortm...,famili diner buffet eclect assort larg chicken...
3,5,Positive,"Wow! Yummy, different, delicious. Our favo...",wow yummi differ delici favorit lamb curri kor...
4,4,Positive,Cute interior and owner (?) gave us tour of up...,cute interior owner gave us tour upcom patioro...
...,...,...,...,...
99995,4,Positive,Came here for lunch with a group. They were bu...,came lunch group busi still room us servic goo...
99996,1,Negative,The equipment is so old and so felty! I just u...,equip old felti upgrad multi club membership c...
99997,4,Positive,This is one of my favorite Mexican restaurants...,one favorit mexican restaur authent menu typic...
99998,2,Negative,Came here for brunch - had an omlette ($19 + t...,came brunch omlett tax tip food wayyyyyyy over...


In [ ]:
# Split the data into train and test sets (80% train, 20% test)
x = df['processed_text']
y = df['label']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# Shape of train data set
print('x_train shape : ', x_train.shape)
print('y_train shape : ', y_train.shape)

x_train shape :  (80000,)
y_train shape :  (80000,)


In [ ]:
# Shape of test data set
print('x_test shape : ', x_test.shape)
print('y_test shape : ', y_test.shape)

x_test shape :  (20000,)
y_test shape :  (20000,)


In [ ]:
def plot_confusion_matrix(cm):

    # Plot Confusion Matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

In [ ]:
def plot_roc_curve(y_true, y_pred):
    # Convert 'Positive' to 1 and 'Negative' to 0 for both true and predicted values
    y_true_modified = y_true.map({'Negative': 0, 'Positive': 1})
    y_pred_modified = np.where(y_pred == 'Positive', 1, 0)

    # Compute ROC curve and ROC area
    fpr, tpr, _ = roc_curve(y_true_modified, y_pred_modified)
    roc_auc = auc(fpr, tpr)

    # Plot the ROC curve
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.show()

In [ ]:
# Custom transformer for tokenizing text
class TextTokenizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [text.split() for text in X]  # Simple unigram tokenization (split by space)

In [87]:
# Custom transformer for one-hot encoding with sparse output and vocabulary filtering
class OneHotEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=5):
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
        self.min_frequency = min_frequency
        self.vocab = None

    def fit(self, X, y=None):
        # Flatten and count token frequencies
        all_tokens = [token for doc in X for token in doc]
        token_counts = Counter(all_tokens)
        self.vocab = {token for token, count in token_counts.items() if count >= self.min_frequency}
        filtered_tokens = [[token for token in doc if token in self.vocab] for doc in X]
        flat_features = [[token] for doc in filtered_tokens for token in doc]
        self.encoder.fit(flat_features)
        return self

    def transform(self, X):
        filtered_tokens = [[token for token in doc if token in self.vocab] for doc in X]
        encoded_docs = []
        for doc in filtered_tokens:
            if doc:  # Check if doc is not empty
                encoded_doc = self.encoder.transform([[token] for token in doc]).sum(axis=0)
            else:
                # Create an empty encoded document with the correct shape
                encoded_doc = np.zeros((1, self.encoder.categories_[0].shape[0]))  # Assuming single feature
                encoded_doc = encoded_doc.astype(int)
            encoded_docs.append(encoded_doc)

        return np.vstack(encoded_docs)

In [88]:

# Plot confusion matrix
def plot_confusion_matrix(cm):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

In [89]:
# Main pipeline definition
onehot_pipeline = make_pipeline(
    TextTokenizer(),
    OneHotEncoderTransformer(min_frequency=10),  # Set min_frequency to reduce vocabulary size
    MultinomialNB()
)

In [90]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df['processed_text'], df['label'], test_size=0.2, random_state=42)

In [ ]:
# Train the pipeline
start_time = time.time()
onehot_pipeline.fit(X_train, y_train)
end_time = time.time()

In [ ]:
# Predictions on test and train data
y_pred_train = onehot_pipeline.predict(X_train)
y_pred_test = onehot_pipeline.predict(X_test)

In [ ]:
# Calculate accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred_test)

In [ ]:
# Calculate F1 scores
f1_positive_test = f1_score(y_test, y_pred_test, pos_label='Positive')
f1_negative_test = f1_score(y_test, y_pred_test, pos_label='Negative')

f1_positive_train = f1_score(y_train, y_pred_train, pos_label='Positive')
f1_negative_train = f1_score(y_train, y_pred_train, pos_label='Negative')

In [ ]:
# Confusion matrices
cm_train = confusion_matrix(y_train, y_pred_train)
cm_test = confusion_matrix(y_test, y_pred_test)

In [ ]:
# Output results
print(f"Training Time: {end_time - start_time:.2f} seconds")
print("Train Accuracy:", accuracy_train)
print(f"F1 Score for Positive Class (Train): {f1_positive_train:.4f}")
print(f"F1 Score for Negative Class (Train): {f1_negative_train:.4f}")
print("Confusion Matrix (Train):\n", cm_train)
print("\nTest Accuracy:", accuracy_test)
print(f"F1 Score for Positive Class (Test): {f1_positive_test:.4f}")
print(f"F1 Score for Negative Class (Test): {f1_negative_test:.4f}")
print("Confusion Matrix (Test):\n", cm_test)


In [ ]:
# Visualize confusion matrix
plot_confusion_matrix(cm_test)

Unigrams with max_features

In [ ]:
# Custom tokenizer for unigram tokenization
class TextTokenizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [text.split() for text in X]  # Tokenize into unigrams

In [ ]:
# Custom transformer for one-hot encoding with max_features
class OneHotEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=1000):
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
        self.max_features = max_features
        self.vocab = None

    def fit(self, X, y=None):
        # Flatten and count token frequencies
        all_tokens = [token for doc in X for token in doc]
        token_counts = Counter(all_tokens)
        # Keep only the top `max_features` tokens
        most_common = token_counts.most_common(self.max_features)
        self.vocab = {token for token, _ in most_common}
        filtered_tokens = [[token for token in doc if token in self.vocab] for doc in X]
        flat_features = [[token] for doc in filtered_tokens for token in doc]
        self.encoder.fit(flat_features)
        return self

    def transform(self, X):
        filtered_tokens = [[token for token in doc if token in self.vocab] for doc in X]
        encoded_docs = [self.encoder.transform([[token] for token in doc]).sum(axis=0) for doc in filtered_tokens]
        return np.vstack(encoded_docs)

In [ ]:
# Plot confusion matrix
def plot_confusion_matrix(cm):
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

In [ ]:
# Define the pipeline with max_features
onehot_pipeline = make_pipeline(
    TextTokenizer(),
    OneHotEncoderTransformer(max_features=1000),  # Limit to top 1000 tokens
    MultinomialNB()
)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df['processed_text'], df['label'], test_size=0.2, random_state=42)

In [ ]:
# Train the pipeline
start_time = time.time()
onehot_pipeline.fit(X_train, y_train)
end_time = time.time()

In [ ]:
# Predictions on test and train data
y_pred_train = onehot_pipeline.predict(X_train)
y_pred_test = onehot_pipeline.predict(X_test)


In [ ]:
# Calculate accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred_test)

In [ ]:
# Calculate F1 scores
f1_positive_test = f1_score(y_test, y_pred_test, pos_label='Positive')
f1_negative_test = f1_score(y_test, y_pred_test, pos_label='Negative')

f1_positive_train = f1_score(y_train, y_pred_train, pos_label='Positive')
f1_negative_train = f1_score(y_train, y_pred_train, pos_label='Negative')

In [ ]:
# Confusion matrices
cm_train = confusion_matrix(y_train, y_pred_train)
cm_test = confusion_matrix(y_test, y_pred_test)

In [ ]:
# Output results
print(f"Training Time: {end_time - start_time:.2f} seconds")
print("Train Accuracy:", accuracy_train)
print(f"F1 Score for Positive Class (Train): {f1_positive_train:.4f}")
print(f"F1 Score for Negative Class (Train): {f1_negative_train:.4f}")
print("Confusion Matrix (Train):\n", cm_train)
print("\nTest Accuracy:", accuracy_test)
print(f"F1 Score for Positive Class (Test): {f1_positive_test:.4f}")
print(f"F1 Score for Negative Class (Test): {f1_negative_test:.4f}")
print("Confusion Matrix (Test):\n", cm_test)

In [ ]:
# Visualize confusion matrix
plot_confusion_matrix(cm_test)

Code for Unigram + Bigram with max_features

In [ ]:
# Custom tokenizer for unigram + bigram tokenization
class UnigramBigramTokenizer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        tokenized_docs = []
        for text in X:
            tokens = text.split()
            unigrams = tokens  # Unigrams
            bigrams = [' '.join(bigram) for bigram in ngrams(tokens, 2)]  # Bigrams
            tokenized_docs.append(unigrams + bigrams)
        return tokenized_docs

In [ ]:

# Custom transformer for one-hot encoding with max_features
class OneHotEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=1000):
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
        self.max_features = max_features
        self.vocab = None

    def fit(self, X, y=None):
        # Flatten and count token frequencies
        all_tokens = [token for doc in X for token in doc]
        token_counts = Counter(all_tokens)
        # Keep only the top `max_features` tokens
        most_common = token_counts.most_common(self.max_features)
        self.vocab = {token for token, _ in most_common}
        filtered_tokens = [[token for token in doc if token in self.vocab] for doc in X]
        flat_features = [[token] for doc in filtered_tokens for token in doc]
        self.encoder.fit(flat_features)
        return self

    def transform(self, X):
        filtered_tokens = [[token for token in doc if token in self.vocab] for doc in X]
        encoded_docs = [self.encoder.transform([[token] for token in doc]).sum(axis=0) for doc in filtered_tokens]
        return np.vstack(encoded_docs)

In [ ]:
# Plot confusion matrix
def plot_confusion_matrix(cm):
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()


In [ ]:
# Define the pipeline for unigram + bigram processing
unibigram_pipeline = make_pipeline(
    UnigramBigramTokenizer(),
    OneHotEncoderTransformer(max_features=1000),  # Limit to top 1000 features
    MultinomialNB()
)


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df['processed_text'], df['label'], test_size=0.2, random_state=42)


In [ ]:
# Train the pipeline
start_time = time.time()
unibigram_pipeline.fit(X_train, y_train)
end_time = time.time()

In [ ]:
# Predictions on test and train data
y_pred_train = unibigram_pipeline.predict(X_train)
y_pred_test = unibigram_pipeline.predict(X_test)

In [ ]:
# Calculate accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
accuracy_test = accuracy_score(y_test, y_pred_test)


In [ ]:
# Calculate F1 scores
f1_positive_test = f1_score(y_test, y_pred_test, pos_label='Positive')
f1_negative_test = f1_score(y_test, y_pred_test, pos_label='Negative')

f1_positive_train = f1_score(y_train, y_pred_train, pos_label='Positive')
f1_negative_train = f1_score(y_train, y_pred_train, pos_label='Negative')


In [ ]:
# Confusion matrices
cm_train = confusion_matrix(y_train, y_pred_train)
cm_test = confusion_matrix(y_test, y_pred_test)


In [ ]:
# Output results
print(f"Training Time: {end_time - start_time:.2f} seconds")
print("Train Accuracy:", accuracy_train)
print(f"F1 Score for Positive Class (Train): {f1_positive_train:.4f}")
print(f"F1 Score for Negative Class (Train): {f1_negative_train:.4f}")
print("Confusion Matrix (Train):\n", cm_train)
print("\nTest Accuracy:", accuracy_test)
print(f"F1 Score for Positive Class (Test): {f1_positive_test:.4f}")
print(f"F1 Score for Negative Class (Test): {f1_negative_test:.4f}")
print("Confusion Matrix (Test):\n", cm_test)

In [ ]:
# Visualize confusion matrix
plot_confusion_matrix(cm_test)